# Imports

In [14]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v


PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [15]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine            import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools          import Nevergrad_Spice_Bode_Optimizer
from symxplorer.designer_tools.utils    import Frequency_Weight
from symxplorer.designer_tools.domains  import Project_Setup
from symxplorer.designer_tools.tf_models import Second_Order_BP_TF, cascade_tf

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer")
logger.info("!!! Spicelib_Wrapper imported successfully !!!")

07:36:49 - SymXplorer: [INFO] !!! Spicelib_Wrapper imported successfully !!!


# Instantiations


In [16]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
logger = setup_loggers()

07:36:49 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
07:36:49 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-09-23_07-36-49.log
07:36:49 - SymXplorer: [INFO] 🔧 spicelib logger set to DEBUG


In [17]:
# s = sp.symbols("s")
# target_tf = (s + 1) / (s**2 + 24*s + 2)
fc=1e9
q=10
k_bp=1e3
filter_inst = Second_Order_BP_TF(q=q, fc=fc, k_bp=k_bp)
target_tf   = filter_inst.get_tf()
target_tf

200000000000.0*pi*s/(s**2 + 200000000.0*pi*s + 4.0e+18*pi**2)

In [18]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

07:36:49 - SymXplorer.domains: [INFO] Loaded project 'Tunable-TIA' with 8 DUT params and 2 probes.


Project_Setup(project_name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(9.999999999999999e-06), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(9.999999999999999e-06), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(9.999999999999999e-06), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(9.999999999999999e-06

In [19]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.project_name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

07:36:49 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
07:36:50 - SymXplorer.spicelib: [INFO] --------------------------------------------------
07:36:50 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
07:36:50 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
07:36:50 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
07:36:50 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
07:36:50 - SymXplorer.spicelib: [INFO] --------------------------------------------------
07:36:50 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
07:36:50 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
07:36:50 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
07:36:50 - SymXplorer.spicelib: [INFO] Te

In [20]:
circuit_optimizer = Nevergrad_Spice_Bode_Optimizer(
    spicelib_wrapper=wrapper,
    target_tf=target_tf,
    output_node='vout',
    frequency_weight=Frequency_Weight(lower=fc/10, upper=fc*10),
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

# Sanity Check

In [21]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

07:36:50 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
07:36:50 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
07:36:50 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
07:36:50 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
07:36:50 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
07:36:50 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [22]:
circuit_optimizer.parameterize()

Dict(x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Log{Cl(0,6,b),exp=2.15},x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 10.000000000000002, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0}

In [23]:
circuit_optimizer.optimize()

07:36:50 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 100
Optimizing:   0%|          | 0/100 [00:00<?, ?trial/s]2025-09-23 07:36:50,866 - nevergrad.optimization.optimizerlib - CMA selected CMAbounded optimizer.
07:36:51 - SymXplorer.optimizer: [INFO] computing the target complex response for 200000000000.0*pi*s/(s**2 + 200000000.0*pi*s + 4.0e+18*pi**2)
Optimizing: 100%|██████████| 100/100 [00:37<00:00,  2.69trial/s]


[{'params': {'x_dut_nfet_w': 8.583638272669553,
   'x_dut_nfet_l': 44.52510129333675,
   'x_dut_cap_w': 54.32742369839054,
   'x_dut_cap_l': 44.70428086357543,
   'x_dut_res_s_l': 60.10734002488238,
   'x_dut_res_s_w': 49.71590450092246,
   'x_dut_res_3_l': 47.452482792152054,
   'x_dut_res_3_w': 47.29036301544533},
  'loss': np.float64(56297.96881336615)},
 {'params': {'x_dut_nfet_w': 8.540364199420422,
   'x_dut_nfet_l': 56.41494624907126,
   'x_dut_cap_w': 60.2458874612616,
   'x_dut_cap_l': 53.502959901263466,
   'x_dut_res_s_l': 52.03294209927277,
   'x_dut_res_s_w': 46.38660414769597,
   'x_dut_res_3_l': 44.800016324100724,
   'x_dut_res_3_w': 51.14963024817535},
  'loss': np.float64(56888.168077225535)},
 {'params': {'x_dut_nfet_w': 7.734155551958918,
   'x_dut_nfet_l': 52.75682067837801,
   'x_dut_cap_w': 70.12559281010888,
   'x_dut_cap_l': 44.39531143372941,
   'x_dut_res_s_l': 37.05362836348356,
   'x_dut_res_s_w': 41.81315389801435,
   'x_dut_res_3_l': 42.888078589065636,
 

In [24]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

07:37:28 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
07:37:28 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [25]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss = out

07:37:28 - SymXplorer.optimizer: [INFO] best loss: 1437.5176749433497


In [26]:
circuit_optimizer.plot_solution(best_param)

07:37:28 - SymXplorer.optimizer: [INFO] total loss: 1437.5176749433497
07:37:28 - SymXplorer.optimizer: [INFO] mag_loss 194.11468913246253, phase_loss 14501.804162989425
